# 예제 01. 토큰 실측과 보정 · Example 01 — Measuring Tokens and Calibrating an Estimate

전산실험 2 · AI Agent 만들기 · **2주차** · 예상 소요 30분
Computational Experiments 2 · Building AI Agents · **Week 2** · about 30 minutes

## 목표 · Objectives
- 응답 객체의 `usage` 세 숫자를 읽는다
- **한국어가 영어보다 토큰을 얼마나 더 쓰는지** 직접 측정한다
- 글자 수에서 토큰 수를 예측하는 **자기만의 근사식**을 회귀로 만든다
- 근사식의 오차를 확인하고, 예산에 얼마나 여유를 둬야 할지 정한다

> **EN**
> - Read the three numbers in the response object's `usage`
> - Measure for yourself **how many more tokens Korean uses than English**
> - Fit a regression to build **your own approximation** from characters to tokens
> - Check the error of that approximation and decide how much budget slack to leave

> 호출은 50회 남짓입니다. 대부분 `max_tokens=1` 로 아주 짧게 부릅니다.
> **EN** — About 50 calls, most of them very short at `max_tokens=1`.

### 준비 · Setup

2주차는 아직 `kmu_llm.py` 를 만들기 전입니다 (예제 03 에서 만듭니다).
그래서 1주차와 같은 방식으로 접속 함수를 이 노트북 안에 직접 정의합니다.

서버 주소와 API 키는 **코드에 적지 않습니다.** Colab Secrets 의
`KMU_BASE_URL`, `KMU_API_KEY` 에서 읽습니다. 왼쪽 열쇠 아이콘(🔑) 에 저장하고
**노트북 액세스 토글을 켜야** 읽힙니다.

> **EN** — In week 2 we have not built `kmu_llm.py` yet (that is Example 03), so
> the connection helpers are defined inline here, exactly as in week 1.
>
> The server URL and the API key are **never written in code**: they come from
> the Colab secrets `KMU_BASE_URL` and `KMU_API_KEY`. Save them under the key
> icon (🔑) on the left and **turn on notebook access**, or they cannot be read.

In [ ]:
!pip install -q openai

import os
from openai import OpenAI

def get_secret(name, secret=True):
    # Colab Secrets -> 환경변수 -> 직접 입력. 값은 코드에 적지 않는다.
    # EN — Colab Secrets -> environment variable -> prompt. Never write the value here.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
MODEL    = "gemma4"

client = OpenAI(api_key=get_secret("KMU_API_KEY"), base_url=BASE_URL, timeout=120)


def raw(messages, **kw):
    """응답 객체를 그대로 돌려준다. usage 를 봐야 하므로.

    EN — Return the whole response object, because we need to read `usage`.
    """
    kw.setdefault("temperature", 0.0)
    return client.chat.completions.create(model=MODEL, messages=messages, **kw)


def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    return raw(messages, **kw).choices[0].message.content.strip()

print("연결 준비 완료:", MODEL)

---
## 1. 세 개의 숫자 · Three numbers

지난주에 "`usage` 를 습관적으로 확인하십시오" 라고 했습니다.
오늘은 그 숫자를 해부합니다.

> **EN** — Last week the advice was to check `usage` out of habit. Today we take
> those numbers apart.

In [ ]:
r = raw([{"role": "user", "content": "표준오차를 한 문장으로 설명해 주세요."}],
        max_tokens=100)

print("응답:", r.choices[0].message.content.strip())
print()
print("prompt_tokens     =", r.usage.prompt_tokens,     "  ← 우리가 보낸 것")
print("completion_tokens =", r.usage.completion_tokens, "  ← 모델이 만든 것")
print("total_tokens      =", r.usage.total_tokens)
print("finish_reason     =", r.choices[0].finish_reason)

`prompt_tokens` 는 **100% 우리 책임**입니다. 무엇을 보낼지 우리가 정하니까요.
`completion_tokens` 는 모델이 정하지만 `max_tokens` 와 지시문으로 조절됩니다.

에이전트에서 문제가 되는 쪽은 거의 항상 `prompt_tokens` 입니다.
루프를 돌면서 대화 이력을 계속 다시 보내기 때문입니다.

> **EN** — `prompt_tokens` is **entirely our responsibility**: we decide what to
> send. `completion_tokens` is the model's choice, but `max_tokens` and the
> instructions steer it.
>
> In an agent the trouble is almost always on the `prompt_tokens` side, because
> the loop keeps resending the conversation history.

---
## 2. 빈 문자열을 보내면 토큰이 0일까 · Does an empty string cost zero tokens?

메시지는 그냥 텍스트로 전송되지 않습니다. 역할 표시와 구분자가 함께 붙습니다.
그 **포장 비용**이 몇 토큰인지 재 봅시다.

아래에서는 응답을 받을 필요가 없으므로 `max_tokens=1` 로 부릅니다.
우리가 보려는 것은 `prompt_tokens` 뿐입니다.

> **EN** — A message is not sent as bare text: role markers and delimiters ride
> along with it. Let's measure that **packaging cost** in tokens.
>
> We do not need the reply below, so we call with `max_tokens=1` — the only
> thing we want is `prompt_tokens`.

In [ ]:
def prompt_tokens(text):
    """이 텍스트를 user 메시지로 보내면 입력 토큰이 몇 개인가.

    EN — How many input tokens does this text cost as a user message?
    """
    r = raw([{"role": "user", "content": text}], max_tokens=1)
    return r.usage.prompt_tokens


for t in ["", "가", "가나", "통계", "통계학과 대학원생입니다"]:
    print(f"{len(t):2d}자  {prompt_tokens(t):3d} 토큰   {t!r}")

빈 문자열인데도 토큰이 0이 아닙니다. 이 값이 **포장 비용**이고,
나중에 만들 근사식의 **절편**이 됩니다.

> 이 값은 서버 구현에 따라 다릅니다. 인터넷에서 본 숫자를 믿지 말고
> 방금 여러분이 잰 값을 쓰십시오.

> **EN** — Even the empty string is not free. That figure is the **packaging
> cost**, and it becomes the **intercept** of the approximation we fit later.
>
> The value depends on the server implementation. Do not trust a number you read
> on the internet — use the one you just measured.

---
## 3. 한국어는 영어보다 얼마나 비싼가 · How much more expensive is Korean?

같은 뜻의 문장 20쌍을 준비했습니다. 각각을 보내 입력 토큰을 재고 비교합니다.

번역 품질이 들쭉날쭉하면 비교가 지저분해지므로 목록을 미리 맞춰 두었습니다.

> **EN** — Twenty sentence pairs of matched meaning. We send each one, measure
> the input tokens and compare. The list is pre-aligned, because uneven
> translation quality would muddy the comparison.

In [ ]:
쌍 = [
    ("두 그룹의 평균을 비교하시오.",              "Compare the means of the two groups."),
    ("이 변수는 연속형입니다.",                   "This variable is continuous."),
    ("정규성 가정을 검토하시오.",                 "Check the normality assumption."),
    ("결측값을 어떻게 처리할까요?",               "How should we handle missing values?"),
    ("표본 크기가 충분하지 않습니다.",            "The sample size is not sufficient."),
    ("유의수준은 0.05로 설정합니다.",             "The significance level is set to 0.05."),
    ("상관관계가 인과관계를 뜻하지는 않습니다.",  "Correlation does not imply causation."),
    ("등분산 가정이 기각되었습니다.",             "The equal variance assumption was rejected."),
    ("사후검정을 수행해야 합니다.",               "A post-hoc test should be performed."),
    ("이상치를 제거할지 판단하시오.",             "Decide whether to remove the outliers."),
    ("회귀계수의 신뢰구간을 구하시오.",           "Compute the confidence interval of the coefficient."),
    ("범주형 변수는 더미로 변환합니다.",          "Categorical variables are converted to dummies."),
    ("검정력이 낮으면 표본을 늘립니다.",          "If power is low, increase the sample size."),
    ("분산분석의 전제 조건을 확인하시오.",        "Check the assumptions of the analysis of variance."),
    ("이 결과는 통계적으로 유의합니다.",          "This result is statistically significant."),
    ("대응표본과 독립표본을 구분하시오.",         "Distinguish paired samples from independent samples."),
    ("효과크기를 함께 보고해야 합니다.",          "The effect size should be reported as well."),
    ("자유도는 표본 크기에서 1을 뺍니다.",        "The degrees of freedom are the sample size minus one."),
    ("비모수 검정을 고려해 보십시오.",            "Consider using a nonparametric test."),
    ("다중비교 보정이 필요합니다.",               "A correction for multiple comparisons is needed."),
]

print(f"{len(쌍)}쌍 준비 완료 — 호출 {len(쌍) * 2}회, 1분 남짓 걸립니다.")

In [ ]:
import pandas as pd

기록 = []
for i, (ko, en) in enumerate(쌍, 1):
    기록.append({"쌍": i, "언어": "ko", "문장": ko,
                "글자수": len(ko), "토큰": prompt_tokens(ko)})
    기록.append({"쌍": i, "언어": "en", "문장": en,
                "글자수": len(en), "토큰": prompt_tokens(en)})
    print(".", end="")

df = pd.DataFrame(기록)
print("\n측정 완료")
df.head(6)

In [ ]:
요약 = df.groupby("언어")[["글자수", "토큰"]].mean().round(1)
요약["글자당토큰"] = (요약["토큰"] / 요약["글자수"]).round(3)
print(요약)

ko평균 = df[df["언어"] == "ko"]["토큰"].mean()
en평균 = df[df["언어"] == "en"]["토큰"].mean()
print(f"\n같은 뜻에 대한 배율:  한국어 / 영어 = {ko평균 / en평균:.2f} 배")

### 무엇이 보이는가 · What you should see

- 한국어가 영어보다 토큰을 더 씁니다. 배율은 대개 1.5 ~ 2.5 사이입니다.
- **글자당 토큰**을 보면 차이가 더 뚜렷합니다. 한글 한 글자와 영문 한 글자의
  값이 다릅니다. 그래서 근사식에 계수를 따로 둬야 합니다.

왜 이런 차이가 나는지는 슬라이드에서 다룹니다. 어휘집이 영어 위주로
만들어졌기 때문입니다.

> **EN**
> - Korean uses more tokens than English, typically by a factor of 1.5 to 2.5.
> - **Tokens per character** makes the gap even clearer: one Hangul character and
>   one Latin character are worth different amounts, which is why the
>   approximation needs a separate coefficient for each.
>
> The slides explain why: the vocabulary was built around English.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for 언어, 색 in [("ko", "tab:red"), ("en", "tab:blue")]:
    부분 = df[df["언어"] == 언어]
    ax.scatter(부분["글자수"], 부분["토큰"], label=언어, color=색, alpha=0.8)

ax.set_xlabel("characters")
ax.set_ylabel("prompt tokens")
ax.set_title("Characters vs tokens, by language")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. 근사식을 회귀로 만든다 · Fitting the approximation by regression

토큰 수를 미리 알고 싶은데, 호출해 봐야만 알 수 있다면 소용이 없습니다.
**보내기 전에** 예측할 수 있어야 예산 관리가 됩니다.

글자 종류별 개수를 설명변수로 두고 회귀를 적합시킵니다.

```
토큰 ≈ 절편 + a × (한글 글자수) + b × (그 외 글자수)
tokens ≈ intercept + a × (Hangul characters) + b × (other characters)
```

절편은 앞에서 본 포장 비용입니다.

> **EN** — Knowing the token count only after you call is no use; budgeting
> requires predicting it **before you send**. So we regress the token count on
> the character counts by type. The intercept is the packaging cost we just saw.

In [ ]:
import numpy as np

def 한글수(text):
    # EN — Count the Hangul syllable characters.
    return sum(1 for ch in text if "가" <= ch <= "힣")

df["한글"] = df["문장"].map(한글수)
df["기타"] = df["글자수"] - df["한글"]

# 설계행렬: [1, 한글수, 기타수]
# EN — Design matrix: [1, Hangul count, other count]
X = np.column_stack([np.ones(len(df)), df["한글"], df["기타"]])
y = df["토큰"].to_numpy(dtype=float)

계수, 잔차합, _, _ = np.linalg.lstsq(X, y, rcond=None)
절편, a, b = 계수

예측 = X @ 계수
SS_res = float(((y - 예측) ** 2).sum())
SS_tot = float(((y - y.mean()) ** 2).sum())
R2 = 1 - SS_res / SS_tot

print(f"토큰 ≈ {절편:.2f} + {a:.3f} × 한글글자 + {b:.3f} × 기타글자")
print(f"R² = {R2:.4f}")
print(f"잔차 표준편차 = {np.std(y - 예측, ddof=3):.2f} 토큰")

In [ ]:
def estimate_tokens(text, 절편=절편, a=a, b=b):
    """호출하지 않고 입력 토큰 수를 어림한다.

    EN — Estimate the input token count without making a call.
    """
    한글 = 한글수(text)
    return 절편 + a * 한글 + b * (len(text) - 한글)


for t in ["", "두 그룹의 평균을 비교하시오.", "Compare the means of the two groups."]:
    print(f"추정 {estimate_tokens(t):6.1f}   실제 {prompt_tokens(t):3d}   {t!r}")

In [ ]:
잔차 = y - 예측

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(예측, 잔차, alpha=0.8)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel("fitted")
axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs fitted")
axes[0].grid(alpha=0.3)

axes[1].hist(잔차, bins=12)
axes[1].set_xlabel("residual (tokens)")
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.show()

print(f"잔차 범위: {잔차.min():.1f} ~ {잔차.max():.1f} 토큰")

### 여유분은 얼마로 둘 것인가 · How much slack should you leave?

잔차의 표준편차가 나왔습니다. 예산을 세울 때 그 **2~3배**를 여유분으로
비워 두면 추정이 빗나가도 안전합니다.

```
대화에 쓸 예산 = 컨텍스트 윈도우 − 응답용 max_tokens − 여유분
conversation budget = context window − max_tokens for the reply − slack
```

정확한 값이 꼭 필요하면 호출한 뒤 `usage` 를 보면 됩니다.
다만 예산 관리에는 **근사로 충분합니다.** 우리가 하려는 것은
"이쯤에서 줄여야겠다" 는 결정이지 정확한 회계가 아닙니다.

> **EN** — You now have the residual standard deviation. Leaving **two to three
> times** that as slack keeps you safe when the estimate misses.
>
> If you truly need the exact figure, call and read `usage`. For budgeting,
> though, **an approximation is enough**: the decision we are making is "time to
> trim", not exact accounting.

---
## 확인한 것 · What we confirmed

1. `usage` 는 세 숫자다. 통제 가능한 쪽은 `prompt_tokens` 다.
2. 빈 문자열도 토큰을 쓴다 — 메시지 **포장 비용**이 절편이다.
3. 한국어는 같은 뜻에 토큰을 1.5~2.5배 쓴다.
4. 글자 종류별 계수를 회귀로 구하면 **호출 전에** 토큰을 어림할 수 있다.
5. 근사의 오차를 알면 여유분을 얼마로 둘지 정할 수 있다.

> **EN**
> 1. `usage` holds three numbers; the one you control is `prompt_tokens`
> 2. Even an empty string costs tokens — the message **packaging cost** is the intercept
> 3. Korean spends 1.5–2.5× the tokens for the same meaning
> 4. Regressing on per-type character counts lets you estimate tokens **before calling**
> 5. Knowing the error of the approximation tells you how much slack to leave

**메모해 두세요.** 방금 구한 세 계수는 예제 03 에서 `kmu_llm.py` 에
`estimate_tokens()` 로 넣습니다.

> **EN — note this down.** The three coefficients you just fitted go into
> `kmu_llm.py` as `estimate_tokens()` in Example 03.

---
## 연습문제 · Exercises

1. 전공 용어 10개(회귀, 이분산성, 부트스트랩, 공분산, 가설검정, …)의 토큰 수를
   한국어와 영어로 비교하시오. 문장보다 배율이 커지는가, 작아지는가?
2. 숫자와 특수문자가 많은 문자열(예: `"p=0.0473, t=-2.81, df=118"`)에서
   근사식의 오차가 어떻게 되는지 확인하시오.
3. 설명변수에 "숫자 글자수"를 추가해 회귀를 다시 적합시키고 R² 를 비교하시오.
4. `system` 메시지를 추가하면 포장 비용이 얼마나 늘어나는지 재시오.

> **EN**
> 1. Compare the token counts of ten technical terms (regression,
>    heteroscedasticity, bootstrap, covariance, hypothesis test, …) in Korean and
>    English. Is the ratio larger or smaller than for sentences?
> 2. Check how the approximation's error behaves on strings full of digits and
>    symbols, e.g. `"p=0.0473, t=-2.81, df=118"`.
> 3. Add a "digit character count" predictor, refit, and compare R².
> 4. Measure how much the packaging cost grows when you add a `system` message.

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary>해설 보기 · Show the solutions</summary>

**1.** 단어 단위에서는 배율이 더 커지는 경우가 많습니다. 영어 전공 용어는
어휘집에 통째로 들어 있어 1~2 토큰이지만, 같은 뜻의 한국어는 조각으로
쪼개집니다. 문장에서는 조사와 어미가 반복되면서 배율이 다소 완화됩니다.

**2.** 오차가 커집니다. 숫자와 기호는 글자당 토큰 비용이 한글·영문과 다릅니다.
근사식이 두 종류의 글자만 구분하기 때문에 이 부분을 흡수하지 못합니다.

**3.** 대개 R² 가 올라갑니다. 잔차 그림에서 보이던 패턴이 줄어듭니다.
설명변수를 하나 더 넣어 잔차의 구조를 흡수한 것입니다. 다만 계수가 늘면
과적합 위험도 생기므로, 실제로 쓸 문장의 성격에 맞춰 정하십시오.

**4.** `system` 메시지 하나가 붙으면 그 내용의 토큰에 더해 포장 비용이
한 번 더 듭니다. 그리고 `system` 은 **매 호출마다 다시 보내집니다.**
길게 쓰면 그 비용이 호출 수만큼 곱해집니다.

---

**EN — 1.** At the word level the ratio is often larger. An English technical
term sits in the vocabulary whole and costs 1–2 tokens, while its Korean
equivalent is chopped into pieces. In sentences the ratio softens somewhat,
because particles and endings repeat.

**EN — 2.** The error grows. Digits and symbols cost a different number of tokens
per character than Hangul or Latin letters, and an approximation that
distinguishes only two character classes cannot absorb that.

**EN — 3.** R² usually rises and the pattern visible in the residual plot
shrinks: the extra predictor absorbs structure that was left in the residuals.
More coefficients also mean more overfitting risk, so decide based on the kind of
text you will actually send.

**EN — 4.** Adding one `system` message costs its own content in tokens plus one
more packaging charge. And `system` **is resent on every single call**, so a long
one multiplies that cost by the number of calls.

</details>